# BASIL DistilBERT Bias Classifier

This Colab notebook fine-tunes a small transformer model, `DistilBERT`, on the BASIL corpus for sentence-level bias detection.

Constraints respected:
- No MCP
- No secondary LLM in the primary pass
- Outputs exported to files so a later verifier LLM can consume them


In [1]:
import sys
print(sys.executable)

import transformers, datasets, torch, sklearn, pandas, numpy
print("imports ok")


/home/rwb6928/venvs/basil-hpc/bin/python
imports ok


## Why DistilBERT

- It is a real transformer model.
- It is much better suited to BASIL than trying to train a large decoder LLM from scratch.
- It is lightweight enough to fine-tune in Google Colab.]


## 1. Install dependencies

In [2]:
!pip -q install transformers datasets accelerate evaluate scikit-learn pandas numpy

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## 2. Download BASIL

In [3]:
# Colab Notebook Code
"""
from pathlib import Path
import zipfile
import urllib.request

ROOT = Path('/content')
ARCHIVE_URL = 'https://github.com/marshallwhiteorg/emnlp19-media-bias/archive/refs/heads/master.zip'
ARCHIVE_PATH = ROOT / 'basil_source.zip'
EXTRACT_DIR = ROOT / 'basil_source'
COMBINED_ZIP = EXTRACT_DIR / 'emnlp19-media-bias-master' / 'emnlp19-BASIL.zip'
COMBINED_DIR = ROOT / 'emnlp19-BASIL'
DATASET_DIR = COMBINED_DIR / 'data'

if not ARCHIVE_PATH.exists():
    urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)

if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ARCHIVE_PATH) as zf:
        zf.extractall(EXTRACT_DIR)

if not COMBINED_DIR.exists():
    with zipfile.ZipFile(COMBINED_ZIP) as zf:
        zf.extractall(ROOT)

print('Dataset dir:', DATASET_DIR)
print('JSON files:', len(list(DATASET_DIR.glob('*.json'))))
"""
# Notebook Code (Jupyter)
from pathlib import Path
import zipfile
import urllib.request

ROOT = Path.home() / "basil_workspace"
ROOT.mkdir(parents=True, exist_ok=True)

ARCHIVE_URL = 'https://github.com/marshallwhiteorg/emnlp19-media-bias/archive/refs/heads/master.zip'
ARCHIVE_PATH = ROOT / 'basil_source.zip'
EXTRACT_DIR = ROOT / 'basil_source'
COMBINED_ZIP = EXTRACT_DIR / 'emnlp19-media-bias-master' / 'emnlp19-BASIL.zip'
COMBINED_DIR = ROOT / 'emnlp19-BASIL'
DATASET_DIR = COMBINED_DIR / 'data'

if not ARCHIVE_PATH.exists():
    urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)

if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ARCHIVE_PATH) as zf:
        zf.extractall(EXTRACT_DIR)

if not COMBINED_DIR.exists():
    with zipfile.ZipFile(COMBINED_ZIP) as zf:
        zf.extractall(ROOT)

print('Root:', ROOT)
print('Dataset dir:', DATASET_DIR)
print('JSON files:', len(list(DATASET_DIR.glob('*.json'))))


Root: /home/rwb6928/basil_workspace
Dataset dir: /home/rwb6928/basil_workspace/emnlp19-BASIL/data
JSON files: 300


## 3. Build the BASIL sentence dataset

In [4]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


def load_basil_sentences(dataset_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for path in sorted(dataset_dir.glob('*.json')):
        article = json.loads(path.read_text())
        article_id = path.stem
        event_id, source = article_id.split('_', 1)
        for sentence_obj in article['body']:
            sentence = (sentence_obj.get('sentence') or '').strip()
            annotations = sentence_obj.get('annotations', [])
            bias_types = sorted(
                {
                    annotation.get('bias', '').strip().lower()
                    for annotation in annotations
                    if annotation.get('bias')
                }
            )
            targets = sorted(
                {
                    annotation.get('target', '').strip()
                    for annotation in annotations
                    if annotation.get('target')
                }
            )
            rows.append(
                {
                    'example_id': f"{article_id}::{sentence_obj['sentence-index']}",
                    'event_id': event_id,
                    'article_id': article_id,
                    'source': source.lower(),
                    'date': article.get('date'),
                    'title': article.get('title'),
                    'url': article.get('url'),
                    'main_event': article.get('main-event'),
                    'sentence_index': sentence_obj['sentence-index'],
                    'sentence_text': sentence,
                    'label': int(bool(annotations)),
                    'gold_annotation_count': len(annotations),
                    'gold_bias_types': bias_types,
                    'gold_targets': targets,
                }
            )

    frame = pd.DataFrame(rows)
    frame['sentence_text'] = frame['sentence_text'].fillna('')
    return frame


def split_sentence_frame(frame: pd.DataFrame, test_size: float = 0.2, random_state: int = 42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(frame, frame['label'], groups=frame['event_id']))
    train_frame = frame.iloc[train_idx].reset_index(drop=True)
    test_frame = frame.iloc[test_idx].reset_index(drop=True)
    train_frame['split'] = 'train'
    test_frame['split'] = 'test'
    return train_frame, test_frame


frame = load_basil_sentences(DATASET_DIR)
train_frame, test_frame = split_sentence_frame(frame)

print('All sentences:', len(frame))
print('Train sentences:', len(train_frame))
print('Test sentences:', len(test_frame))
print('Train positive rate:', round(train_frame['label'].mean(), 4))
print('Test positive rate:', round(test_frame['label'].mean(), 4))

All sentences: 7984
Train sentences: 6327
Test sentences: 1657
Train positive rate: 0.2036
Test positive rate: 0.2034


## 4. Convert to Hugging Face datasets

In [5]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_frame, preserve_index=False)
test_dataset = Dataset.from_pandas(test_frame, preserve_index=False)
dataset_dict = DatasetDict({'train': train_dataset, 'test': test_dataset})
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['example_id', 'event_id', 'article_id', 'source', 'date', 'title', 'url', 'main_event', 'sentence_index', 'sentence_text', 'label', 'gold_annotation_count', 'gold_bias_types', 'gold_targets', 'split'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['example_id', 'event_id', 'article_id', 'source', 'date', 'title', 'url', 'main_event', 'sentence_index', 'sentence_text', 'label', 'gold_annotation_count', 'gold_bias_types', 'gold_targets', 'split'],
        num_rows: 1657
    })
})

## 5. Tokenizer and model

In [8]:
import numpy as np
import evaluate

from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def tokenize_batch(batch):
    return tokenizer(batch['sentence_text'], truncation=True, padding='max_length', max_length=256)

tokenized = dataset_dict.map(tokenize_batch, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

accuracy_metric = evaluate.load('accuracy')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    metrics = {}
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(precision_metric.compute(predictions=preds, references=labels, zero_division=0))
    metrics.update(recall_metric.compute(predictions=preds, references=labels, zero_division=0))
    metrics.update(f1_metric.compute(predictions=preds, references=labels))
    return metrics

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/6327 [00:00<?, ? examples/s]

Map:   0%|          | 0/1657 [00:00<?, ? examples/s]

## 6. Fine-tune DistilBERT

In [18]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

OUTPUT_DIR = ROOT / 'distilbert_basil_model'
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
    fp16=True,
    logging_steps=25,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.167800,0.778025,0.801448,0.518692,0.329377,0.402904
2,0.148600,0.944165,0.770670,0.433022,0.412463,0.422492
3,0.103800,1.257674,0.777912,0.447458,0.391691,0.417722
4,0.119500,1.281445,0.794810,0.493213,0.323442,0.390681
5,0.056700,1.403284,0.802052,0.522613,0.308605,0.388060
6,0.004700,1.509287,0.794810,0.492823,0.305638,0.377289
7,0.016200,1.587716,0.790585,0.479675,0.350148,0.404803
8,0.000300,1.611592,0.794206,0.491453,0.341246,0.402802


TrainOutput(global_step=3168, training_loss=0.0688808450410366, metrics={'train_runtime': 250.9865, 'train_samples_per_second': 201.668, 'train_steps_per_second': 12.622, 'total_flos': 3352484925186048.0, 'train_loss': 0.0688808450410366, 'epoch': 8.0})

## 7. Evaluate on held-out BASIL sentences

In [19]:
eval_metrics = trainer.evaluate()
eval_metrics

{'eval_loss': 0.9441654682159424,
 'eval_accuracy': 0.7706698853349426,
 'eval_precision': 0.43302180685358255,
 'eval_recall': 0.4124629080118694,
 'eval_f1': 0.42249240121580545,
 'eval_runtime': 2.1198,
 'eval_samples_per_second': 781.673,
 'eval_steps_per_second': 24.53,
 'epoch': 8.0}

## 8. Create verifier-ready output files

In [20]:
import json
import numpy as np
from scipy.special import softmax

EXPORT_DIR = ROOT / 'outputs' / 'distilbert_basil'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

raw_predictions = trainer.predict(tokenized['test'])
logits = raw_predictions.predictions
probabilities = softmax(logits, axis=1)
predicted_labels = np.argmax(logits, axis=1)

export_frame = test_frame.copy()
export_frame['predicted_has_bias'] = predicted_labels
export_frame['predicted_bias_probability'] = probabilities[:, 1]
export_frame['prediction_confidence'] = np.max(probabilities, axis=1)
export_frame['primary_model'] = MODEL_NAME

metrics_path = EXPORT_DIR / 'metrics.json'
heldout_path = EXPORT_DIR / 'heldout_predictions.csv'
summary_path = EXPORT_DIR / 'article_summary.csv'
jsonl_path = EXPORT_DIR / 'sentence_predictions.jsonl'

metrics_payload = {
    'model_name': MODEL_NAME,
    'train_sentence_count': int(len(train_frame)),
    'test_sentence_count': int(len(test_frame)),
    'positive_rate_train': float(train_frame['label'].mean()),
    'positive_rate_test': float(test_frame['label'].mean()),
    'metrics': {key: float(value) for key, value in eval_metrics.items() if isinstance(value, (int, float))},
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2))

with jsonl_path.open('w') as handle:
    for row in export_frame.to_dict(orient='records'):
        record = {
            'example_id': row['example_id'],
            'event_id': row['event_id'],
            'article_id': row['article_id'],
            'source': row['source'],
            'date': row['date'],
            'title': row['title'],
            'url': row['url'],
            'main_event': row['main_event'],
            'sentence_index': int(row['sentence_index']),
            'sentence_text': row['sentence_text'],
            'split': row['split'],
            'primary_model': row['primary_model'],
            'primary_prediction': {
                'label': 'biased' if int(row['predicted_has_bias']) == 1 else 'not_biased',
                'bias_probability': round(float(row['predicted_bias_probability']), 6),
                'confidence': round(float(row['prediction_confidence']), 6),
            },
            'gold_reference': {
                'has_bias': int(row['label']),
                'annotation_count': int(row['gold_annotation_count']),
                'bias_types': row['gold_bias_types'],
                'targets': row['gold_targets'],
            },
        }
        handle.write(json.dumps(record) + '\n')

export_frame.to_csv(heldout_path, index=False)

article_summary = (
    export_frame.groupby(['event_id', 'article_id', 'source', 'title', 'date', 'url', 'main_event'], dropna=False)
    .agg(
        sentence_count=('example_id', 'count'),
        gold_biased_sentences=('label', 'sum'),
        predicted_biased_sentences=('predicted_has_bias', 'sum'),
        mean_bias_probability=('predicted_bias_probability', 'mean'),
        max_bias_probability=('predicted_bias_probability', 'max'),
    )
    .reset_index()
    .sort_values(['event_id', 'source'])
    .reset_index(drop=True)
)
article_summary.to_csv(summary_path, index=False)

print('Wrote:', metrics_path)
print('Wrote:', heldout_path)
print('Wrote:', summary_path)
print('Wrote:', jsonl_path)

Wrote: /home/rwb6928/basil_workspace/outputs/distilbert_basil/metrics.json
Wrote: /home/rwb6928/basil_workspace/outputs/distilbert_basil/heldout_predictions.csv
Wrote: /home/rwb6928/basil_workspace/outputs/distilbert_basil/article_summary.csv
Wrote: /home/rwb6928/basil_workspace/outputs/distilbert_basil/sentence_predictions.jsonl


In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd

rows = []
for threshold in [i / 100 for i in range(10, 91, 5)]:
    preds = (export_frame["predicted_bias_probability"] >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(export_frame["label"], preds),
        "precision": precision_score(export_frame["label"], preds, zero_division=0),
        "recall": recall_score(export_frame["label"], preds, zero_division=0),
        "f1": f1_score(export_frame["label"], preds, zero_division=0),
        "predicted_positive_rate": preds.mean(),
    })

threshold_results = pd.DataFrame(rows).sort_values("f1", ascending=False)
threshold_results


best = threshold_results.iloc[0]
print("Best:", best)

Best: threshold                  0.350000
accuracy                   0.764635
precision                  0.424929
recall                     0.445104
f1                         0.434783
predicted_positive_rate    0.213036
Name: 5, dtype: float64


## 9. Preview outputs

In [12]:
print(metrics_path.read_text())
with jsonl_path.open() as handle:
    for _ in range(2):
        print(next(handle).strip())

{
  "model_name": "distilbert-base-uncased",
  "train_sentence_count": 6327,
  "test_sentence_count": 1657,
  "positive_rate_train": 0.20357199304567725,
  "positive_rate_test": 0.20337960168980085,
  "metrics": {
    "eval_loss": 0.5394281148910522,
    "eval_accuracy": 0.8020519010259505,
    "eval_precision": 0.51931330472103,
    "eval_recall": 0.3590504451038576,
    "eval_f1": 0.4245614035087719,
    "eval_runtime": 2.9202,
    "eval_samples_per_second": 567.423,
    "eval_steps_per_second": 17.807,
    "epoch": 3.0
  }
}
{"example_id": "0_fox::0", "event_id": "0", "article_id": "0_fox", "source": "fox", "date": "2012-08-15", "title": "Ryan goes on offense over Medicare, accuses Obama of treating program like 'piggy bank'", "url": "http://www.foxnews.com/politics/2012/08/14/ryan-defends-gop-record-on-medicare-says-party-has-plan-to-save-program/?utm_source=feedburner&utm_medium=feed&utm_campaign=Feed%3A+foxnews%2Fpolitics+%28Internal+-+Politics+-+Text%29", "main_event": "Obama an